### CSU CM 515 (Tai Montgomery, 2/25/2026)
# 🧬 Databases and Genome Browsers
In this notebook, we'll explore genome information databases and genome browsers.

### 🧩 Exercise 1: Retrieving sequence data.
Here we'll explore several sequence databases.

1.1. GenBank is a comprehensive database of nucleotide sequences and their protein translations. Go to the [GenBank website](https://www.ncbi.nlm.nih.gov/genbank/). Search for `drosha`.  Explore the output and learn how to get the human gene sequence in fasta format.

1.2. The Gene Expression Omnibus (GEO) is the major US repository for NGS data. Go to the [GEO website](https://www.ncbi.nlm.nih.gov/geo/). Search for GSE133898 and explore the contents. View one of the samples in the SRA viewer. 

We won't explore it today, but [SRA toolkit](https://github.com/ncbi/sra-tools/wiki/01.-Downloading-SRA-Toolkit) makes it simple to download NGS datasets. 

### 🧩  Exercise 2: Exploring organismal databases and web-based genome browsers
Here we'll explore organismal databases.

2.1. **WormBase** is the database of C. elegans and other nematodes. It contains vast information on every gene in the worm. Let's explore [WormBase](https://www.wormbase.org/#01-23-6) together.

2.2. **ensembl** is a great source of genome information for numerous species.  I'll demonstrate how to download the human genome sequence and annotations.  Navigate to the [ensembl website](http://www.ensembl.org/Homo_sapiens/Info/Index?redirect=no).

2.3. The **UCSC Genome Browser** contains many different genomes and is one of the most commonly used genome browsers. Go to the [UCSC Genome Browser](https://genome.ucsc.edu/index.html) and navigate to the human genome.

2.4. Find the button `Hide all` and click on it.

2.5. Find the button `GENCODE V49` and select `full`. Then click on any of the `refresh` buttons.

2.6. In the `Expression` category, select `full` for `GTEx RNA-Seq Coverage`. Then click on a `refresh` button.

2.7. Search for the gene `YTHDF2` and answer the following questions:
  * What are the chromosome and genomic coordinates of the gene?
  * How many alternative isoforms are there and what makes them different?
  * What is the orientation (strand) of the gene?
  * What tissue has the highest expression in the GTEx data?

### 🧩 Exercise 3: Visualizing data in IGV
Here we'll use IGV to visualize the E. coli alignment data from Monday. Download and install IGV [here](https://igv.org).

3.1. First we'll need to install `samtools` so we prepare the `sam` files for loading into IGV:

In [ ]:
%%bash
conda install -c bioconda -c conda-forge samtools

3.2. Using `samtools`, we'll convert the `sam` files from Monday to `bam` format.

In [ ]:
samtools view -bS 'input.sam' > 'output.bam'

3.3. Next we'll use `samtools` to sort the `bam` files by genomic coordinates.

In [ ]:
%%bash
samtools sort 'input_file' -o 'input_file.sorted.bam'

3.4. Lastly we need to create an index for the sorted bam file.

In [ ]:
%%bash
samtools index 'input_file.sorted.bam'

3.5. Now that the files are correctly formatted, launch [IGV](https://igv.org).

3.6. From the genome bar, select the E. coli genome (U00096.2).

3.7. From the menu bar, select `File` and navigate to `Load from file`. Navigate to your sorted `bam` files and select them.

3.8. While visualizing the data explore the options for changing track height and color, displaying reads, and navigating the genome.  Here are some questions to consider:
 * How many chromsomes does E. coli have?
 * Why are the reads not uniformly distributed?
 * How does the read distribution differ in the data filtered for perfect matches?  

3.9. Convert BAM files to count files (.tdf) using the `igvtools` `count` function (`Tools` > `Run igvtools...` > navigate to `sorted.bam` file.

3.10. Load the `.tdf` files as in step 3.7.

3.11. Save the IGV session so it can be viewed later: `File` > `Save session`.

### 🧩 Exercise 4: Parsing GFF3 files with Python
Here we'll use python to calculate the average number of exons per transcript in humans and worms.

4.1. Use the `curl` command to download GFF3s for humans and worms (if you don't have `curl`, you can copy and paste the web links in your browser to download the files.

In [ ]:
%%bash
curl -L -O https://ftp.ensembl.org/pub/current_gff3/homo_sapiens/Homo_sapiens.GRCh38.115.gff3.gz
curl -L -O https://ftp.ensemblgenomes.ebi.ac.uk/pub/release-60/metazoa/gff3/caenorhabditis_elegans/Caenorhabditis_elegans.WBcel235.60.gff3.gz

If needed, decompress the files using `gzip`:

In [ ]:
gzip -d 'file_name.gz'

4.2. Use the Python script below to calculate the average numbers of exons in humans and worms.  How do they differ?

In [ ]:
import sys

gff3_file = sys.argv[1]

transcripts = set()          # all transcript IDs (mRNA IDs)
exon_count = {}              # transcript_id -> number of exons

for line in open(gff3_file):
    if line.startswith("#"):
        continue

    # Unpack each line in the gff3 file into variables using str.split()
    seq, source, feature, start, end, score, strand, frame, notes = line.rstrip().split("\t")

    # Save every transcript ID
    if feature == "mRNA":
        transcript_id = notes.split("ID=")[1].split(";")[0]
        transcripts.add(transcript_id)

    # Count exons for each transcript
    if feature == "exon":
        transcript_id = notes.split("Parent=")[1].split(";")[0]
        exon_count[transcript_id] = exon_count.get(transcript_id, 0) + 1

# Compute average exons per transcript
total_exons = sum(exon_count.get(t, 0) for t in transcripts)
average = f"{total_exons / len(transcripts):.2f}"

print("Average exons per transcript:", average)

#### `scp` command

In [ ]:
scp username@server:/path/to/remote/file /path/to/local/destination/